# Plans: reject wrong analyses before they run

In GIS-agent benchmarks up to ~47% of failed runs involve planning mistakes (missing or mis-ordered operations). MapSmith turns those silent runtime failures into **machine-actionable errors before anything executes**: an agent submits a typed plan, static validation checks operations, arguments, references, input files and simulated CRS flow.

This is the `validate_plan` / `execute_plan` MCP tool pair, used here as a library.

In [1]:
import json
from pathlib import Path

import geopandas as gpd
from shapely.geometry import Point, box

data = Path('data'); data.mkdir(exist_ok=True)
wells = gpd.GeoDataFrame(
    {'name': ['well_a', 'well_b', 'well_c', 'well_d']},
    geometry=[Point(9.19, 45.46), Point(9.21, 45.475),
              Point(9.17, 45.45), Point(9.23, 45.49)], crs='EPSG:4326')
wells.to_file(data / 'wells.gpkg')
zone = gpd.GeoDataFrame({'zone': ['center']},
                        geometry=[box(9.16, 45.44, 9.22, 45.48)], crs='EPSG:4326')
zone.to_parquet(data / 'zone.parquet')
print('inputs ready')

inputs ready


## A wrong plan (on purpose)

Two classic agent mistakes: a typo in an operation name, and a step consuming the output of a *later* step (the mis-ordering failure class).

In [2]:
from mapsmith.plans import Plan, validate

wrong = Plan.model_validate({'goal': 'wells at risk', 'steps': [
    {'id': 'cut', 'operation': 'clip_layer', 'arguments': {
        'input_path': '$buf',          # <- refers to a LATER step
        'mask_path': str(data / 'zone.parquet'),
        'output_path': str(data / 'at_risk.parquet')}},
    {'id': 'buf', 'operation': 'bufffer',   # <- typo
     'arguments': {'input_path': str(data / 'wells.gpkg'),
                   'distance_meters': 500,
                   'output_path': str(data / 'wells_500m.parquet')}},
]})
report = validate(wrong)
print('valid:', report.valid)
for e in report.errors:
    print(f'  [{e.code}] step {e.step_id}: {e.message}')

valid: False
  [FORWARD_REFERENCE] step cut: 'input_path' references '$buf' which runs later — move step 'buf' before 'cut'
  [UNKNOWN_OPERATION] step buf: operation 'bufffer' does not exist. Did you mean: buffer_layer?


Stable error codes, the exact step, and a suggestion — an agent can repair this plan without a human.

## The corrected plan: validate, then execute

In [3]:
good = Plan.model_validate({'goal': 'wells at risk', 'steps': [
    {'id': 'buf', 'operation': 'buffer_layer',
     'arguments': {'input_path': str(data / 'wells.gpkg'),
                   'distance_meters': 500,
                   'output_path': str(data / 'wells_500m.parquet')}},
    {'id': 'cut', 'operation': 'clip_layer',
     'arguments': {'input_path': '$buf',
                   'mask_path': str(data / 'zone.parquet'),
                   'output_path': str(data / 'at_risk.parquet')}},
]})
report = validate(good)
print('valid:', report.valid)
for sid, sim in report.simulated_outputs.items():
    print(f'  step {sid} -> {sim.output} (simulated CRS: {sim.crs})')
for n in report.notes:
    print(f'  note [{n.code}]: {n.message}')

valid: True
  step buf -> data\wells_500m.parquet (simulated CRS: EPSG:4326)
  step cut -> data\at_risk.parquet (simulated CRS: EPSG:4326)
  note [CRS_NOTE]: input is in a geographic CRS: the engine buffers via an estimated UTM zone and records the decision in provenance


In [4]:
from mapsmith.plans import execute

result = execute(good)
print('executed:', result['executed'])
for step in result['steps']:
    print(f"  {step['id']}: {step['status']} "
          f"({step.get('feature_count', '-')} features, "
          f"{step['elapsed_ms']} ms, verified={step.get('verified')})")
print('plan sha256:', result['plan_sha256'][:16], '…')
print('plan manifest:', result['plan_manifest'])

executed: True
  buf: ok (4 features, 225.4 ms, verified=True)
  cut: ok (3 features, 73.4 ms, verified=True)
plan sha256: a5d2cadb11b7baf8 …
plan manifest: data\at_risk.parquet.plan.json


In [5]:
manifest = json.loads(Path(result['plan_manifest']).read_text())
print(json.dumps({k: manifest[k] for k in
                  ('mapsmith_version', 'plan_sha256', 'goal')}, indent=2))
print('steps:', [(s['id'], s['status']) for s in manifest['steps']])

{
  "mapsmith_version": "0.4.0",
  "plan_sha256": "a5d2cadb11b7baf8722806fb7aca608c4ab9e22ac64f019e438c680f6b5ac852",
  "goal": "wells at risk"
}
steps: [('buf', 'ok'), ('cut', 'ok')]


The plan-level manifest fingerprints the exact plan that produced the result; each step keeps its own provenance manifest. Reproducibility is a hash away.